In [5]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [6]:
files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(files)} CSV files")

for file in files:
    print(file.name)

Found 18 CSV files
account_status_history.csv
accounts.csv
agent_sessions.csv
agents.csv
borrowers.csv
call_attempts.csv
call_dispositions.csv
calls.csv
campaigns.csv
complaints.csv
daily_targeting.csv
data_dictionary.csv
field_visits.csv
payments.csv
promises_to_pay.csv
sms_events.csv
vendor_telephony.csv
whatsapp_events.csv


In [7]:
data = {}

for file in files:
    data[file.stem] = pd.read_csv(file)

print("Loaded datasets:")
print(list(data.keys()))

Loaded datasets:
['account_status_history', 'accounts', 'agent_sessions', 'agents', 'borrowers', 'call_attempts', 'call_dispositions', 'calls', 'campaigns', 'complaints', 'daily_targeting', 'data_dictionary', 'field_visits', 'payments', 'promises_to_pay', 'sms_events', 'vendor_telephony', 'whatsapp_events']


In [8]:
inventory = []

for name, df in data.items():
    inventory.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        )
    })

inventory = pd.DataFrame(inventory)

inventory = inventory.sort_values(
    "rows",
    ascending=False
).reset_index(drop=True)

display(inventory)

,dataset,rows,columns,duplicate_rows,memory_mb
0,call_attempts,120000,9,0,56.13
1,calls,91350,11,1271,52.35
2,whatsapp_events,60600,8,600,28.41
3,account_status_history,60000,8,0,27.73
4,sms_events,45000,8,0,20.86
5,daily_targeting,45000,7,0,15.21
6,call_dispositions,35000,8,0,16.08
7,borrowers,30600,8,600,12.98
8,accounts,30000,11,0,13.79
9,agents,30000,8,0,13.73


In [9]:
inventory.to_csv(
    REPORT_DIR / "dataset_inventory.csv",
    index=False
)

In [10]:
schema_rows = []

for name, df in data.items():
    for col in df.columns:
        schema_rows.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "rows": len(df),
            "null_count": int(df[col].isna().sum()),
            "null_pct": round(df[col].isna().mean() * 100, 2),
            "unique_count": int(df[col].nunique(dropna=False)),
            "unique_pct": round(
                df[col].nunique(dropna=False) / len(df) * 100,
                2
            )
        })

schema = pd.DataFrame(schema_rows)

display(schema)

,dataset,column,dtype,rows,null_count,null_pct,unique_count,unique_pct
0,account_status_history,history_id,object,60000,0,0.0,60000,100.00
1,account_status_history,account_id,object,60000,0,0.0,25999,43.33
2,account_status_history,borrower_id,object,60000,0,0.0,11916,19.86
3,account_status_history,event_at,object,60000,0,0.0,59898,99.83
4,account_status_history,status,object,60000,0,0.0,7,0.01
...,...,...,...,...,...,...,...,...
141,whatsapp_events,event_at,object,60600,0,0.0,59892,98.83
142,whatsapp_events,message_id,object,60600,0,0.0,34831,57.48
143,whatsapp_events,event_type,object,60600,0,0.0,6,0.01
144,whatsapp_events,template_code,object,60600,0,0.0,5,0.01


In [11]:
schema.to_csv(
    REPORT_DIR / "schema_profile.csv",
    index=False
)

In [12]:
candidate_keys = schema[
    (schema["unique_pct"] >= 99.9) &
    (schema["null_count"] == 0)
].copy()

candidate_keys = candidate_keys.sort_values(
    ["dataset", "unique_pct"],
    ascending=[True, False]
)

display(candidate_keys)

,dataset,column,dtype,rows,null_count,null_pct,unique_count,unique_pct
0,account_status_history,history_id,object,60000,0,0.0,60000,100.00
8,accounts,account_id,object,30000,0,0.0,30000,100.00
11,accounts,principal_amount,float64,30000,0,0.0,29996,99.99
12,accounts,outstanding_amount,float64,30000,0,0.0,29994,99.98
16,accounts,opened_at,object,30000,0,0.0,29993,99.98
19,agent_sessions,session_id,object,15000,0,0.0,15000,100.00
21,agent_sessions,login_at,object,15000,0,0.0,14996,99.97
25,agent_sessions,logout_at,object,15000,0,0.0,14996,99.97
32,agents,joined_at,object,30000,0,0.0,29995,99.98
33,agents,updated_at,object,30000,0,0.0,29987,99.96


In [13]:
duplicates = []

for name, df in data.items():
    duplicates.append({
        "dataset": name,
        "total_rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_pct": round(
            df.duplicated().mean() * 100,
            2
        )
    })

duplicates = pd.DataFrame(duplicates)

duplicates = duplicates.sort_values(
    "duplicate_rows",
    ascending=False
)

display(duplicates)

,dataset,total_rows,duplicate_rows,duplicate_pct
7,calls,91350,1271,1.39
4,borrowers,30600,600,1.96
17,whatsapp_events,60600,600,0.99
13,payments,25500,486,1.91
1,accounts,30000,0,0.00
0,account_status_history,60000,0,0.00
5,call_attempts,120000,0,0.00
6,call_dispositions,35000,0,0.00
3,agents,30000,0,0.00
2,agent_sessions,15000,0,0.00


In [14]:
duplicates.to_csv(
    REPORT_DIR / "duplicate_report.csv",
    index=False
)

In [15]:
missing = []

for name, df in data.items():
    for col in df.columns:
        null_count = int(df[col].isna().sum())

        missing.append({
            "dataset": name,
            "column": col,
            "null_count": null_count,
            "null_pct": round(
                null_count / len(df) * 100,
                2
            )
        })

missing = pd.DataFrame(missing)

missing = missing.sort_values(
    "null_pct",
    ascending=False
)

display(missing.head(50))

,dataset,column,null_count,null_pct
37,borrowers,email,895,2.92
36,borrowers,phone,614,2.01
63,calls,agent_id,1827,2.00
49,call_attempts,vendor_id,2400,2.00
9,accounts,borrower_id,455,1.52
110,payments,payment_reference,382,1.50
105,field_visits,scheduled_at,250,1.00
2,account_status_history,borrower_id,0,0.00
1,account_status_history,account_id,0,0.00
8,accounts,account_id,0,0.00


In [16]:
missing.to_csv(
    REPORT_DIR / "missing_values.csv",
    index=False
)

In [17]:
date_columns = []

for name, df in data.items():
    for col in df.columns:
        if any(
            x in col.lower()
            for x in ["date", "time", "timestamp", "at"]
        ):
            date_columns.append({
                "dataset": name,
                "column": col
            })

date_columns = pd.DataFrame(date_columns)

display(date_columns)

,dataset,column
0,account_status_history,event_at
1,account_status_history,status
2,account_status_history,recorded_at
3,accounts,status
4,accounts,opened_at
5,accounts,timezone
6,agent_sessions,login_at
7,agent_sessions,timezone
8,agent_sessions,logout_at
9,agents,status


In [18]:
date_ranges = []

for name, df in data.items():

    for col in df.columns:

        if not any(
            x in col.lower()
            for x in ["date", "time", "timestamp", "at"]
        ):
            continue

        parsed = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        date_ranges.append({
            "dataset": name,
            "column": col,
            "min_date": parsed.min(),
            "max_date": parsed.max(),
            "invalid_dates": int(parsed.isna().sum())
        })

date_ranges = pd.DataFrame(date_ranges)

display(date_ranges)

C:\Users\ridhi\AppData\Local\Temp\ipykernel_10052\2705481916.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
C:\Users\ridhi\AppData\Local\Temp\ipykernel_10052\2705481916.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
C:\Users\ridhi\AppData\Local\Temp\ipykernel_10052\2705481916.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
C:\Users\ridhi\AppData\Local\Temp\ipykernel_10052\2705481916.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back 

,dataset,column,min_date,max_date,invalid_dates
0,account_status_history,event_at,2026-01-01 00:01:08.000000000,2026-08-08 23:50:45.000000000,0
1,account_status_history,status,NaT,NaT,60000
2,account_status_history,recorded_at,2025-12-31 01:26:29.000000000,2026-08-09 22:02:27.000000000,0
3,accounts,status,NaT,NaT,30000
4,accounts,opened_at,2024-01-01 00:02:27.000000000,2025-11-30 23:52:36.000000000,0
5,accounts,timezone,NaT,NaT,30000
6,agent_sessions,login_at,2026-01-01 00:01:57.000000000,2026-08-08 23:51:54.000000000,0
7,agent_sessions,timezone,NaT,NaT,15000
8,agent_sessions,logout_at,2026-01-01 04:48:48.000000000,2026-08-09 07:50:12.000000000,0
9,agents,status,NaT,NaT,30000


In [19]:
date_ranges.to_csv(
    REPORT_DIR / "date_ranges.csv",
    index=False
)

In [20]:
categorical_values = []

for name, df in data.items():

    for col in df.columns:

        if df[col].dtype == "object":
            unique_count = df[col].nunique(dropna=False)

            if unique_count <= 30:
                values = df[col].value_counts(
                    dropna=False
                ).to_dict()

                categorical_values.append({
                    "dataset": name,
                    "column": col,
                    "unique_count": unique_count,
                    "values": values
                })

categorical_values = pd.DataFrame(categorical_values)

display(categorical_values)

,dataset,column,unique_count,values
0,account_status_history,status,7,"{'PAID': 8650, 'CLOSED': 8614, 'DELINQUENT': 8..."
1,account_status_history,source,5,"{'CORE': 12079, 'BATCH': 12073, 'CALL': 12028,..."
2,accounts,loan_type,5,"{'CREDIT_CARD': 6080, 'AUTO': 6079, 'PERSONAL'..."
3,accounts,risk_segment,4,"{'HIGH': 7552, 'MEDIUM': 7533, 'LOW': 7513, 'N..."
4,accounts,status,4,"{'ACTIVE': 7539, 'CLOSED': 7496, 'PAID': 7486,..."
5,accounts,timezone,3,"{'UTC': 10096, 'Asia/Kolkata': 9981, 'Asia/Dub..."
6,accounts,schema_version,3,"{'v1': 10152, 'v2': 10026, 'v3': 9822}"
7,agent_sessions,channel,4,"{'WHATSAPP': 3810, 'FIELD': 3796, 'VOICE': 374..."
8,agent_sessions,timezone,2,"{'Asia/Kolkata': 7506, 'UTC': 7494}"
9,agents,agent_name,10,"{'Sneha Das': 3151, 'Amit Kumar': 3119, 'Priya..."


In [21]:
id_columns = []

for name, df in data.items():

    for col in df.columns:

        if "id" in col.lower():
            id_columns.append({
                "dataset": name,
                "column": col,
                "unique_values": df[col].nunique(dropna=False),
                "nulls": df[col].isna().sum()
            })

id_columns = pd.DataFrame(id_columns)

display(id_columns)

,dataset,column,unique_values,nulls
0,account_status_history,history_id,60000,0
1,account_status_history,account_id,25999,0
2,account_status_history,borrower_id,11916,0
3,accounts,account_id,30000,0
4,accounts,borrower_id,10944,455
5,agent_sessions,session_id,15000,0
6,agent_sessions,agent_id,1000,0
7,agent_sessions,device_id,1500,0
8,agents,agent_id,1000,0
9,agents,vendor_id,15,0


In [22]:
accounts = data["accounts"]
borrowers = data["borrowers"]

missing_borrowers = accounts[
    ~accounts["borrower_id"].isin(
        borrowers["borrower_id"]
    )
]

print("Accounts with invalid/missing borrower:")
print(len(missing_borrowers))

Accounts with invalid/missing borrower:
2913


In [23]:
calls = data["calls"]

invalid_call_accounts = calls[
    ~calls["account_id"].isin(
        accounts["account_id"]
    )
]

print(
    "Calls with invalid account:",
    len(invalid_call_accounts)
)

Calls with invalid account: 0


In [24]:
payments = data["payments"]

invalid_payment_accounts = payments[
    ~payments["account_id"].isin(
        accounts["account_id"]
    )
]

print(
    "Payments with invalid account:",
    len(invalid_payment_accounts)
)

Payments with invalid account: 0


In [25]:
agents = data["agents"]

invalid_call_agents = calls[
    calls["agent_id"].notna() &
    ~calls["agent_id"].isin(agents["agent_id"])
]

print(
    "Calls with invalid agent:",
    len(invalid_call_agents)
)

Calls with invalid agent: 0


In [26]:
phase1_summary = {
    "datasets": len(data),
    "total_rows": sum(len(df) for df in data.values()),
    "tables_with_duplicates": int(
        (duplicates["duplicate_rows"] > 0).sum()
    ),
    "tables_with_missing_values": int(
        (missing["null_count"] > 0).groupby(
            missing["dataset"]
        ).any().sum()
    )
}

phase1_summary

{'datasets': 18,
 'total_rows': 639328,
 'tables_with_duplicates': 4,
 'tables_with_missing_values': 6}